In [1]:
!pip install plotly

In [2]:
!pip install "anywidget>=0.9.13"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 3.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.3/477.3 kB 7.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 3.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 3.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 17.9 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install -U kaleido

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.1 MB/s eta 0:00:00:00:0100:01


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import plotly.express as px

In [2]:
df_NN = pd.DataFrame(columns = ['org','celltype','mapping','frac cells'])

In [3]:
a = 0

In [4]:
sm = load_samap('../../Active_SAMap_Joined/sm_hypo_nonneural_07262026.pkl')

In [5]:
ref = 'mg'

In [6]:
mappings = {'316 Bergmann NN':'339 Astrocyte-like NN',
'317 Astro-CB NN':'339 Astrocyte-like NN',
'318 Astro-NT NN':'339 Astrocyte-like NN',
'319 Astro-TE NN':'339 Astrocyte-like NN',
'320 Astro-OLF NN':'339 Astrocyte-like NN',
'334 Microglia NN':'340 Macrophage NN',
'335 BAM NN':'340 Macrophage NN'}

In [7]:
fin_ref = []
for item in sm.sams[ref].adata.obs['subclass_id_label']:
    if item in mappings:
        fin_ref.append(mappings[item])
    else:
        fin_ref.append(item)

In [8]:
sm.sams[ref].adata.obs['figure2e_mapping'] = fin_ref

In [24]:
org = 'cj'

In [25]:
ref_level = 'figure2e_mapping'
org_level = 'ss_subclass_v4_nounlabeled_nn'

In [26]:
keys = {ref:ref_level,org:org_level}
D,MappingTable = get_mapping_scores(sm,keys)

lim_MappingTable = MappingTable.filter(like=org)
lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

/scratch/miniconda/lib/python3.7/site-packages/samap/analysis.py:1609: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  samap.adata.obs[l] = pd.Categorical(cl)


In [27]:
sam = SAM()
sam.load_data('../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad')

In [28]:
for item in sm.sams[ref].adata.obs[ref_level].unique():
    print(org + '_' + item)
    if org + '_' + item in lim_MappingTable.columns:
        df_NN.loc[a, 'mapping'] = lim_MappingTable.loc[ref + '_' + item, org + '_' + item]
        df_NN.loc[a, 'org'] = org
        df_NN.loc[a, 'celltype'] = item
        df_NN.loc[a, 'num cells'] = len(sm.sams[org].adata[sm.sams[org].adata.obs[org_level] == item])
        df_NN.loc[a, 'frac cells'] = len(sm.sams[org].adata[sm.sams[org].adata.obs[org_level] == item])/len(sam.adata)
        a += 1

cj_339 Astrocyte-like NN
cj_322 Tanycyte NN
cj_321 Astroependymal NN
cj_325 CHOR NN
cj_330 VLMC NN
cj_323 Ependymal NN
cj_324 Hypendymal NN
cj_326 OPC NN
cj_327 Oligo NN
cj_329 ABC NN
cj_328 OEC NN
cj_331 Peri NN
cj_332 SMC NN
cj_333 Endo NN
cj_338 Lymphoid NN
cj_340 Macrophage NN
cj_337 DC NN
cj_336 Monocytes NN


In [29]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,dr,339 Astrocyte-like NN,0.673483,0.044734,2748.0
1,dr,325 CHOR NN,0.271209,0.001807,111.0
2,dr,330 VLMC NN,0.862881,0.005128,315.0
3,dr,323 Ependymal NN,0.750009,0.011558,710.0
4,dr,326 OPC NN,0.854567,0.016197,995.0
5,dr,327 Oligo NN,0.738923,0.01216,747.0
6,dr,329 ABC NN,0.607487,0.003305,203.0
7,dr,328 OEC NN,0.441959,0.010972,674.0
8,dr,331 Peri NN,0.787093,0.006723,413.0
9,dr,333 Endo NN,0.843582,0.024678,1516.0


In [30]:
df_NN[df_NN['celltype'] == '330 VLMC NN']

,org,celltype,mapping,frac cells,num cells
4,cj,330 VLMC NN,0.927916,0.006274,469.0
15,ac,330 VLMC NN,0.948212,0.011816,566.0
26,xt,330 VLMC NN,0.853306,0.002392,101.0
35,dr,330 VLMC NN,0.862881,0.005128,315.0


In [31]:
df_NN[df_NN['celltype'] == '333 Endo NN']

,org,celltype,mapping,frac cells,num cells
9,cj,333 Endo NN,0.940984,0.001003,75.0
21,ac,333 Endo NN,0.953661,0.001378,66.0
30,xt,333 Endo NN,0.934827,0.001563,66.0
42,dr,333 Endo NN,0.843582,0.024678,1516.0


In [32]:
df_NN[df_NN['celltype'] == '329 ABC NN']

,org,celltype,mapping,frac cells,num cells
8,cj,329 ABC NN,0.899007,0.001953,146.0
20,ac,329 ABC NN,0.979552,0.000564,27.0
29,xt,329 ABC NN,0.536848,0.000805,34.0
39,dr,329 ABC NN,0.607487,0.003305,203.0


In [33]:
df_NN[df_NN['celltype'] == '340 Macrophage NN']

,org,celltype,mapping,frac cells,num cells
10,cj,340 Macrophage NN,0.871302,0.003103,232.0
23,ac,340 Macrophage NN,0.895612,0.018622,892.0
32,xt,340 Macrophage NN,0.72603,0.002913,123.0
44,dr,340 Macrophage NN,0.506172,0.025932,1593.0


In [34]:
set.intersection(*(set(g['celltype']) for _, g in df_NN.groupby('org')))

{'326 OPC NN',
 '327 Oligo NN',
 '329 ABC NN',
 '330 VLMC NN',
 '333 Endo NN',
 '339 Astrocyte-like NN',
 '340 Macrophage NN'}

In [30]:
ct_int = ['322 Tanycyte NN','326 OPC NN','327 Oligo NN','339 Astrocyte-like NN','340 Macrophage NN']

In [31]:
df_NN = df_NN[df_NN['celltype'].isin(ct_int)]

In [32]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,dr,339 Astrocyte-like NN,0.673483,0.044734,2748.0
4,dr,326 OPC NN,0.854567,0.016197,995.0
5,dr,327 Oligo NN,0.738923,0.01216,747.0
11,dr,340 Macrophage NN,0.506172,0.025932,1593.0
13,xt,339 Astrocyte-like NN,0.535237,0.014803,625.0
14,xt,322 Tanycyte NN,0.573513,0.003529,149.0
16,xt,326 OPC NN,0.842175,0.020772,877.0
17,xt,327 Oligo NN,0.870144,0.012932,546.0
21,xt,340 Macrophage NN,0.72603,0.002913,123.0
22,ac,339 Astrocyte-like NN,0.866155,0.034697,1662.0


In [33]:
test = [0,0.05,0.1]
org = []
fin_ct = []
frac = []
mapping = []
numcells = []
for th in test:
    for item in df_NN['celltype'].unique():
        org.append('test_' + str(th))
        fin_ct.append(item)
        frac.append(th)
        mapping.append(1)
        numcells.append(0)

In [34]:
test_df = pd.DataFrame([org,fin_ct,mapping,frac,numcells], index = ['org','celltype','mapping','frac cells','num cells']).T

In [35]:
test_df

,org,celltype,mapping,frac cells,num cells
0,test_0,339 Astrocyte-like NN,1,0,0
1,test_0,326 OPC NN,1,0,0
2,test_0,327 Oligo NN,1,0,0
3,test_0,340 Macrophage NN,1,0,0
4,test_0,322 Tanycyte NN,1,0,0
5,test_0.05,339 Astrocyte-like NN,1,0.05,0
6,test_0.05,326 OPC NN,1,0.05,0
7,test_0.05,327 Oligo NN,1,0.05,0
8,test_0.05,340 Macrophage NN,1,0.05,0
9,test_0.05,322 Tanycyte NN,1,0.05,0


In [36]:
df_NN = pd.concat([df_NN,test_df])

In [37]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,dr,339 Astrocyte-like NN,0.673483,0.044734,2748.0
4,dr,326 OPC NN,0.854567,0.016197,995.0
5,dr,327 Oligo NN,0.738923,0.01216,747.0
11,dr,340 Macrophage NN,0.506172,0.025932,1593.0
13,xt,339 Astrocyte-like NN,0.535237,0.014803,625.0
14,xt,322 Tanycyte NN,0.573513,0.003529,149.0
16,xt,326 OPC NN,0.842175,0.020772,877.0
17,xt,327 Oligo NN,0.870144,0.012932,546.0
21,xt,340 Macrophage NN,0.72603,0.002913,123.0
22,ac,339 Astrocyte-like NN,0.866155,0.034697,1662.0


In [38]:
df_NN['frac cells'] = df_NN['frac cells'].clip(upper=.1)

In [39]:
df_NN['frac cells'] = df_NN['frac cells'].astype('float')
df_NN['mapping'] = df_NN['mapping'].astype('float')

In [40]:
fig = px.scatter(df_NN, x = 'org', y = 'celltype', size = 'frac cells', color = 'mapping', color_continuous_scale= 'Blues', range_color=[0,.99],opacity = 1)
fig.update_layout(font=dict(family="Helvetica, Arial, sans-serif", color="black"))
fig.update_xaxes(categoryorder='array', categoryarray= ['cj','ac','xt','dr'])
fig.update_yaxes(categoryorder='array', categoryarray= ['322 Tanycyte NN','340 Macrophage NN','339 Astrocyte-like NN','326 OPC NN','327 Oligo NN',])
fig.update_layout(
    autosize=False,
    width=700,
    height=300,
)

fig.write_image("../../Figures/Figures_08022026/NN_mapping_allorgs_08042026.pdf")
fig.write_image("../../Figures/Figures_08022026/NN_mapping_allorgs_08042026.png")
fig.write_image("../../Figures/Figures_08022026/NN_mapping_allorgs_08042026.svg")